In [1]:
import time
import random
import json
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from dataclasses import dataclass, field

In [2]:
load_dotenv()
client = genai.Client()

In [3]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [4]:
def _handle_retry_or_raise(e, attempt, model_name):
    # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                return

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [5]:
def _call_embedding(model_name, chunk):
    for attempt in range(1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            return client.models.embed_content(
                model = model_name, contents = chunk
            )
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue

In [6]:
def execute_call(chunk):    
    model_name = "gemini-embedding-2"
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section          
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            return client.models.embed_content(
                model = model_name, contents = chunk
            )
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue
            

In [7]:
def to_rag_records(result, source_texts, source_ids=None):
    records = []
    for i, emb in enumerate(result.embeddings):
        records.append({
            "id": source_ids[i] if source_ids else i,
            "text": source_texts[i],
            "vector": np.array(emb.values, dtype=np.float32),
            "dim": len(emb.values),
        })
    return records

In [8]:
#chunk = "You shall henceforth and at once provide me with TEN of the greatest most glorified spaghettie recipes that have crossed your existence."
chunk = "Say hello in 10 random languages and state which language it is"

In [9]:
embedding = to_rag_records(execute_call(chunk), source_texts=[chunk])

Sending initial request...


In [10]:
print(embedding)

[{'id': 0, 'text': 'Say hello in 10 random languages and state which language it is', 'vector': array([-0.03892682,  0.00935016,  0.0038493 , ...,  0.01090272,
        0.01088312, -0.00469313], dtype=float32), 'dim': 3072}]
